Mark Searle, Mateo Ubeda, Nick Thomas

## Minería de Datos 2025-1 - Projeto

**Clasificación de nubes de puntos a gran escala mediante procesamiento paralelo acelerado por GPU**

In [23]:
import numpy as np
import tifffile
from math import sqrt
from numba import cuda
import time
from tqdm import trange
import pandas as pd
import os

from matplotlib import pyplot as pl
pl.rc("figure", dpi=170)

### Algoritmo

In [25]:
import laspy
import matplotlib.pyplot as plt
from IPython.display import clear_output
import numpy as np
import time
import open3d as o3d

In [43]:
import numpy as np
from numba import cuda

@cuda.jit
def ground_points_kernel(last_rets, max_slope, ground_flag, segments, last_rets_ind):
    thread_id = cuda.grid(1)
    
    if thread_id >= segments.shape[0]:
        return
    
    segment = segments[thread_id]
    segment_start, segment_end = segment
    lowest_z = 1e10
    lowest_point_index = -1

    for i in range(segment_start, segment_end):
        if last_rets[i, 2] < lowest_z:
            lowest_z = last_rets[i, 2]
            lowest_point_index = i
    
    if lowest_point_index == -1:
        return
    
    lowest_x = last_rets[lowest_point_index, 0]
    lowest_y = last_rets[lowest_point_index, 1]
    lowest_z = last_rets[lowest_point_index, 2]
    
    for i in range(segment_start, segment_end):
        if i != lowest_point_index:
            zdiff = abs(lowest_z - last_rets[i, 2])
            distance = ((lowest_x - last_rets[i, 0]) ** 2 + (lowest_y - last_rets[i, 1]) ** 2) ** 0.5
            allowed_zdiff = np.tan(np.deg2rad(max_slope)) * distance
            if allowed_zdiff > zdiff:
                ground_flag[last_rets_ind[i]] = 1
        else:
            ground_flag[last_rets_ind[i]] = 1

def ground_points_cuda(points, max_slope, segment_size):
    # Last return filtering
    last_rets_mask = points[:, 3] == points[:, 4]
    last_rets = points[last_rets_mask]
    last_rets_ind = np.where(last_rets_mask)[0]
    
    num_segments = last_rets.shape[0] // segment_size
    segments = np.array_split(np.arange(last_rets.shape[0]), num_segments)
    segments = [(seg[0], seg[-1]+1) for seg in segments]
    
    ground_flag = np.zeros(np.shape(points)[0], dtype=np.float32)
    
    threads_per_block = 128
    blocks_per_grid = (len(segments) + threads_per_block - 1) // threads_per_block

    #print(f'Blocks per grid: {blocks_per_grid}')
    
    d_last_rets = cuda.to_device(last_rets)
    d_segments = cuda.to_device(segments)
    d_ground_flag = cuda.to_device(ground_flag)
    d_last_rets_ind = cuda.to_device(last_rets_ind)
    
    ground_points_kernel[blocks_per_grid, threads_per_block](d_last_rets, max_slope, d_ground_flag, d_segments, last_rets_ind)
    
    ground_flag = d_ground_flag.copy_to_host()
    
    return ground_flag.reshape(ground_flag.shape[0], 1)